# 00 — Data build: from CDC ZIP files to one Parquet file per cohort

**Goal.** Turn the 8 CDC linked birth–infant death cohort ZIPs (2016–2023) into data we can analyze: check each ZIP, document every field, and write one Parquet file per cohort.

**Thesis part.** Step 1 (data build). Every later notebook reads the Parquet files made here. Plan: `plan/analysis_plan.md`, section 2.

**Inputs.** `data/cdc/*PE*CO.zip` (3.3 GB compressed, 80.9 GB of text inside) and the user guides `docs/guides/*_linkedUG.pdf`.

**Outputs.** `reports/profile/<cohort>.json`, `reference/columns.csv`, `reference/layout_differences.md`, `data/parquet/cohort_<year>.parquet` (ignored by Git).

**How to run.** Run all cells. With `REBUILD = False`, cohorts whose Parquet file already exists are skipped. With `REBUILD = True`, everything is rebuilt: about 25 minutes and up to 3 GB of memory.

**Terms.**
- A *cohort file* holds all babies born in one year (the *denominator*) and the infant deaths among them (the *numerator*, deaths in year Y and Y+1).
- *Fixed-width*: each item sits at fixed character positions in a text line; there are no separators.
- *Parquet*: a compressed, column-based file format. Reading one column does not read the others.

## 1. Setup

`REBUILD` is a parameter: the command-line runner (papermill) can set it to `True`.

In [1]:
REBUILD = False

In [3]:
%run functions.ipynb
import collections
import csv
import json
import time
import zipfile
from collections import defaultdict

import polars as pl
import pyarrow.parquet as pq

ZIPS = sorted((ROOT / "data" / "cdc").glob("*PE*CO.zip"))
YEARS = range(2016, 2024)
pl.Config.set_tbl_rows(20); pl.Config.set_tbl_width_chars(180)
[z.name for z in ZIPS]

['2017PE2016CO.zip',
 '2018PE2017CO.zip',
 '2019PE2018CO.zip',
 '2020PE2019CO.zip',
 '2021PE2020CO.zip',
 '2022PE2021CO.zip',
 '2023PE2022CO.zip',
 '2024PE2023CO.zip']

## 2. Profile each ZIP

**What.** Read each ZIP as a stream (no extraction). Count births and U.S.-resident births, join the deaths to their births, and measure missing values for 14 key items.

**Why.** Before we build anything, we check that every death joins to a birth and that the record lengths are as the guide says. This also answers whether the missing values are real or a download error.

**Join key.** `CO_SEQNUM` (positions 365–371) + `CO_YOD` (372–375). The 2017 denominator pads `CO_SEQNUM` with spaces and the numerators with zeros, so we compare it as a number.

**Expected.** 0 unmatched deaths in every cohort; infant mortality rate about 5.4–5.8 per 1,000.

In [4]:
def file_year(name: str) -> int:
    v = int(re.search(r"VS(\d+)LINK", name).group(1))
    return v + 2000 if v < 100 else v


def f(r, a, b):
    return r[a - 1:b].decode("latin-1")


def key(r):
    seq = f(r, 365, 371).strip()
    return (int(seq), f(r, 372, 375)) if seq else None


# (label, start, end, missing codes, positive codes)
FIELDS = [
    ('MAGER', 75, 76, set(), set()),
    ('MEDUC', 124, 124, {'9'}, set()),
    ('DMAR', 120, 120, {' '}, set()),
    ('PRECARE', 224, 225, {'99'}, set()),
    ('CIG_0', 253, 254, {'99'}, set()),
    ('BMI', 283, 286, {'99.9', '    '}, set()),
    ('RF_PDIAB', 313, 313, {'U', ' '}, {'Y'}),
    ('RF_GDIAB', 314, 314, {'U', ' '}, {'Y'}),
    ('RF_PHYPE', 315, 315, {'U', ' '}, {'Y'}),
    ('RF_GHYPE', 316, 316, {'U', ' '}, {'Y'}),
    ('PAY', 435, 435, {'9', ' '}, set()),
    ('COMBGEST', 490, 491, {'99'}, set()),
    ('BRTHWGT', 512, 515, {'9999'}, set()),
    ('AB_NICU', 519, 519, {'U', ' '}, {'Y'}),
]


def profile(zp: Path, outdir: Path) -> dict:
    cohort = int(re.search(r"PE(\d{4})CO", zp.name).group(1))
    z = zipfile.ZipFile(zp)
    names = z.namelist()
    den = [n for n in names if ("DENPUB" in n or "DENOM" in n) and file_year(n) == cohort][0]
    nums = [n for n in names if "NUMPUB" in n and file_year(n) in (cohort, cohort + 1)]
    deaths, num_len = {}, collections.Counter()
    for name in nums:
        with z.open(name) as fh:
            for r in fh:
                num_len[len(r.rstrip(b"\r\n"))] += 1
                if f(r, 9, 12) != str(cohort) or int(f(r, 104, 104)) >= 4:   # DOB_YY, RESTATUS
                    continue
                deaths[key(r)] = int(f(r, 1356, 1358))                       # age at death, days
    t = time.time()
    n = res = matched = neo = 0
    den_len = collections.Counter()
    miss, pos = collections.Counter(), collections.Counter()
    for r in z.open(den):
        n += 1
        if n <= 1000:
            den_len[len(r.rstrip(b"\r\n"))] += 1
        if int(f(r, 104, 104)) >= 4:
            continue
        res += 1
        k = key(r)
        if k and k in deaths:
            matched += 1
            neo += deaths[k] < 28
        for lab, a, b, mc, pc in FIELDS:
            v = f(r, a, b)
            if v in mc: miss[lab] += 1
            if v in pc: pos[lab] += 1
    out = {
        "cohort": cohort, "zip": zp.name, "denominator_member": den,
        "numerator_members": nums, "denominator_record_lengths": dict(den_len),
        "numerator_record_lengths": dict(num_len), "denominator_records": n,
        "us_resident_births": res, "cohort_deaths_in_numerators": len(deaths),
        "deaths_matched_to_births": matched, "unmatched_deaths": len(deaths) - matched,
        "neonatal_deaths_lt28d": neo, "imr_per_1000": round(1000 * matched / res, 3),
        "missing_pct": {k: round(100 * miss[k] / res, 3) for k, *_ in FIELDS},
        "positive_pct": {k: round(100 * pos[k] / res, 3) for k, *_ in FIELDS if pos[k] or k.startswith(("RF_", "AB_"))},
        "seconds": round(time.time() - t),
    }
    outdir.mkdir(parents=True, exist_ok=True)
    (outdir / f"{cohort}.json").write_text(json.dumps(out, indent=2))
    print(cohort, "done", out["imr_per_1000"], out["unmatched_deaths"], flush=True)
    return out


PROFILE = ROOT / "reports" / "profile"
profiles = []
for zp in ZIPS:
    cohort = int(re.search(r"PE(\d{4})CO", zp.name).group(1))
    saved = PROFILE / f"{cohort}.json"
    profiles.append(profile(zp, PROFILE) if REBUILD or not saved.exists() else json.loads(saved.read_text()))
pl.DataFrame([{k: p[k] for k in ("cohort", "denominator_records", "us_resident_births", "deaths_matched_to_births",
                                 "unmatched_deaths", "neonatal_deaths_lt28d", "imr_per_1000")} for p in profiles])

2016 done 5.802 0


2017 done 5.749 0


2018 done 5.63 0


2019 done 5.507 0


2020 done 5.354 0


2021 done 5.449 0


2022 done 5.526 0


2023 done 5.49 0


cohort,denominator_records,us_resident_births,deaths_matched_to_births,unmatched_deaths,neonatal_deaths_lt28d,imr_per_1000
i64,i64,i64,i64,i64,i64,f64
2016,3956112,3945875,22893,0,15185,5.802
2017,3864754,3855500,22167,0,14789,5.749
2018,3801534,3791712,21346,0,14205,5.63
2019,3757582,3747540,20639,0,13690,5.507
2020,3619826,3613647,19346,0,12755,5.354
2021,3669928,3664292,19965,0,12620,5.449
2022,3676029,3667758,20268,0,12948,5.526
2023,3605081,3596017,19743,0,12892,5.49


**Result.** Every cohort has 0 unmatched deaths: each infant death in the numerator files joins to exactly one birth. The infant mortality rate is 5.35–5.80 per 1,000. The resident birth counts equal the CDC user guides (the 2020 guide prints 3,549,826, a typo for 3,619,826).

The missing shares below are real gaps on the certificates, not download errors. They are low (BMI about 2%, diabetes and hypertension 0.1–0.2%), except marital status, which California has not released since 2017.

In [5]:
pl.DataFrame([{"cohort": p["cohort"], **p["missing_pct"]} for p in profiles])

cohort,MAGER,MEDUC,DMAR,PRECARE,CIG_0,BMI,RF_PDIAB,RF_GDIAB,RF_PHYPE,RF_GHYPE,PAY,COMBGEST,BRTHWGT,AB_NICU
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2016,0.0,1.305,0.0,2.866,0.488,2.76,0.096,0.096,0.096,0.096,0.621,0.088,0.027,0.112
2017,0.0,1.285,12.232,2.553,0.493,2.518,0.096,0.096,0.096,0.096,0.605,0.072,0.054,0.089
2018,0.0,1.292,11.994,2.515,0.476,2.265,0.076,0.076,0.076,0.076,0.596,0.066,0.055,0.09
2019,0.0,1.444,11.905,2.262,0.514,2.302,0.088,0.088,0.088,0.088,0.687,0.064,0.073,0.091
2020,0.0,1.432,11.62,2.047,0.396,1.948,0.112,0.112,0.112,0.112,0.714,0.069,0.052,0.105
2021,0.0,1.545,11.471,2.113,0.429,2.002,0.142,0.142,0.142,0.142,0.765,0.082,0.032,0.156
2022,0.0,1.67,11.424,2.16,0.418,2.063,0.18,0.18,0.18,0.18,0.892,0.073,0.028,0.191
2023,0.0,1.792,11.12,1.875,0.38,2.286,0.16,0.16,0.16,0.16,0.735,0.073,0.026,0.182


## 3. Record layout of every year: `reference/columns.csv`

**What.** Parse the layout table of all 8 user guides. For each public field, save its positions, codes and missing codes, and list the fields that change between years.

**Why.** A field can move or be renamed between years. If we used one year's positions for another year, we would read the wrong characters.

**Check.** In every year, the fields and fillers must cover positions 1–1346 (births) and 1–1743 (deaths) with no gap and no overlap. The code stops with an error if not.

In [6]:
REF = ROOT / "reference"
layouts = {}
for c in YEARS:
    rows = [r for r in parse_layout(c) if r["public"]]
    for length in (1346, 1743):
        problems = check_tiling(parse_layout(c), length)
        assert not problems, (c, length, problems)
    layouts[c] = {r["name"]: r for r in rows}

names = sorted({n for L in layouts.values() for n in L},
               key=lambda n: min(L[n]["start"] for L in layouts.values() if n in L))
REF.mkdir(exist_ok=True)
with open(REF / "columns.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["name", "start_2023", "end_2023", "file", "description", "missing_codes_auto",
                "years_present", "position_by_year_if_different", "codes"])
    for n in names:
        ref = layouts[2023].get(n) or next(L[n] for L in layouts.values() if n in L)
        years = [c for c in YEARS if n in layouts[c]]
        pos = {c: (layouts[c][n]["start"], layouts[c][n]["end"]) for c in years}
        differs = {c: p for c, p in pos.items() if p != (ref["start"], ref["end"])}
        w.writerow([n, ref["start"] if n in layouts[2023] else "", ref["end"] if n in layouts[2023] else "",
                    "denominator+numerator" if ref["end"] <= 1346 else "numerator only",
                    ref["description"], ref["missing"],
                    f"{years[0]}-{years[-1]}" if years == list(range(years[0], years[-1] + 1)) else ",".join(map(str, years)),
                    "; ".join(f"{c}:{s}-{e}" for c, (s, e) in differs.items()), ref["codes"]])

# Differences report
lines = ["# Layout differences between cohort years", "",
         "Generated by `notebooks/00_data_build.ipynb` from the CDC user guides in `docs/guides/`. "
         "Every year's layout covers positions 1–1346 (births) and 1–1743 (deaths) with no gap or overlap of fields.", ""]
only = defaultdict(list)
for n in names:
    yrs = tuple(c for c in YEARS if n in layouts[c])
    if yrs != tuple(YEARS):
        only[yrs].append(n)
lines += ["## Fields not present in all 8 years", "", "| Years | Fields |", "|---|---|"]
for yrs, ns in sorted(only.items()):
    lines.append(f"| {yrs[0]}–{yrs[-1]} | {', '.join(f'`{x}`' for x in ns)} |" if len(yrs) > 1 else f"| {yrs[0]} | {', '.join(f'`{x}`' for x in ns)} |")
moved = [n for n in names if len({(layouts[c][n]['start'], layouts[c][n]['end']) for c in YEARS if n in layouts[c]}) > 1]
lines += ["", "## Fields whose position changes", ""]
lines += [f"- `{n}`: " + ", ".join(f"{c} {layouts[c][n]['start']}-{layouts[c][n]['end']}" for c in YEARS if n in layouts[c]) for n in moved] or ["None."]
(REF / "layout_differences.md").write_text("\n".join(lines) + "\n")
print(len(names), "public fields;", sum(len(v) for v in only.values()), "not in all years;", len(moved), "moved")

282 public fields; 31 not in all years; 1 moved


**Result.** All 8 layouts tile both record lengths exactly. `reference/layout_differences.md` lists the fields that exist only in some years and the few fields whose position or name changes. The renamed fields are handled in the next section.

## 4. Build one Parquet file per cohort

**What.** For each ZIP:
- read the births file as a stream, in blocks of 600 MB;
- keep every public field of that year's layout (fillers dropped), as the text codes CDC wrote, with spaces stripped and empty values as null;
- rename the few fields that CDC renamed between years (same positions), so every cohort uses the 2023 names;
- join the death fields from the two numerator files on `CO_SEQNUM` + `CO_YOD`, and add `cohort` and `death` (True = linked infant death).

**Why.** Parquet with zstd compression makes one cohort about 120 MB instead of about 5 GB of text, and DuckDB can read only the columns a query needs.

**Expected.** About 2 minutes and 3 GB of memory per cohort; the code stops if any death does not join to a birth.

In [7]:
OUT_DIR = ROOT / "data" / "parquet"
DEN_LEN = 1346
CHUNK_BYTES = 600_000_000
# Older names -> 2023 names (same positions; checked in reference/layout_differences.md).
RENAME = {"AGED": "AGEDX", "AGER5": "AGER5X", "AGER22": "AGER22X", "UC0D": "UCOD", "CO_DODYY": "CO_YOD"}


def slice_cols(lines: pl.Series, fields: list[dict]) -> pl.DataFrame:
    df = pl.DataFrame({"line": lines})
    return df.select([
        pl.col("line").str.slice(f["start"] - 1, f["end"] - f["start"] + 1)
        .str.strip_chars().replace("", None).alias(f["name"])
        for f in fields
    ])


def key_expr() -> list[pl.Expr]:
    # 2017 pads CO_SEQNUM with spaces, numerators with zeros: compare as integers.
    return [pl.col("CO_SEQNUM").cast(pl.Int64, strict=False).alias("_seq"), pl.col("CO_YOD").alias("_yod")]


def build(zip_path: Path) -> Path:
    cohort = int(re.search(r"PE(\d{4})CO", zip_path.name).group(1))
    fields = [dict(f, name=RENAME.get(f["name"], f["name"])) for f in parse_layout(cohort)
              if f["public"] and not f["name"].upper().startswith("FILLER")]
    den_fields = [f for f in fields if f["end"] <= DEN_LEN]
    death_fields = [f for f in fields if f["start"] > DEN_LEN]

    z = zipfile.ZipFile(zip_path)
    names = z.namelist()
    den_name = next(n for n in names if ("DENPUB" in n or "DENOM" in n) and file_year(n) == cohort)
    num_names = [n for n in names if "NUMPUB" in n and file_year(n) in (cohort, cohort + 1)]

    # Deaths: both numerator files, keep babies born in the cohort year.
    key_fields = [f for f in den_fields if f["name"] in ("DOB_YY", "CO_SEQNUM", "CO_YOD")]
    deaths = []
    for n in num_names:
        lines = pl.Series(z.read(n).decode("latin-1").splitlines())
        d = slice_cols(lines, key_fields + death_fields)
        deaths.append(d.filter(pl.col("DOB_YY") == str(cohort)))
    deaths = pl.concat(deaths).with_columns(key_expr()).drop("DOB_YY", "CO_SEQNUM", "CO_YOD")
    assert deaths.select(pl.struct("_seq", "_yod").is_duplicated().any()).item() is False

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    out = OUT_DIR / f"cohort_{cohort}.parquet"
    deaths = deaths.with_columns(pl.lit(True).alias("death"))
    writer, n_rows, n_linked = None, 0, 0

    def write(body: bytes):
        nonlocal writer, n_rows, n_linked
        lines = pl.Series(body.decode("latin-1").splitlines())
        df = slice_cols(lines, den_fields).with_columns(key_expr())
        # Survivors have a blank CO_SEQNUM (null key), so they never match.
        df = df.join(deaths, on=["_seq", "_yod"], how="left")
        df = df.with_columns(pl.col("death").fill_null(False),
                             pl.lit(cohort, pl.Int16).alias("cohort")).drop("_seq", "_yod")
        n_rows += df.height
        n_linked += int(df["death"].sum())
        table = df.to_arrow()
        if writer is None:
            writer = pq.ParquetWriter(out, table.schema, compression="zstd")
        writer.write_table(table)

    rest = b""
    with z.open(den_name) as fh:
        while True:
            block = fh.read(CHUNK_BYTES)
            if not block:
                if rest.strip():
                    write(rest)
                break
            block = rest + block
            cut = block.rfind(b"\n") + 1
            body, rest = block[:cut], block[cut:]
            if body:
                write(body)
    writer.close()
    unmatched = deaths.height - n_linked
    print(f"{cohort}: births {n_rows:,}  deaths in numerators {deaths.height:,}  linked {n_linked:,}  "
          f"unmatched {unmatched}  -> {out.name} {out.stat().st_size / 1e6:,.0f} MB")
    assert unmatched == 0, "some deaths did not join to a birth"
    return out


for zp in ZIPS:
    cohort = int(re.search(r"PE(\d{4})CO", zp.name).group(1))
    out = OUT_DIR / f"cohort_{cohort}.parquet"
    if REBUILD or not out.exists():
        build(zp)
    else:
        print(f"{cohort}: kept existing {out.name}")

2016: births 3,956,112  deaths in numerators 22,927  linked 22,927  unmatched 0  -> cohort_2016.parquet 126 MB


2017: births 3,864,754  deaths in numerators 22,197  linked 22,197  unmatched 0  -> cohort_2017.parquet 119 MB


2018: births 3,801,534  deaths in numerators 21,379  linked 21,379  unmatched 0  -> cohort_2018.parquet 117 MB


2019: births 3,757,582  deaths in numerators 20,681  linked 20,681  unmatched 0  -> cohort_2019.parquet 117 MB


2020: births 3,619,826  deaths in numerators 19,376  linked 19,376  unmatched 0  -> cohort_2020.parquet 112 MB


2021: births 3,669,928  deaths in numerators 19,982  linked 19,982  unmatched 0  -> cohort_2021.parquet 113 MB


2022: births 3,676,029  deaths in numerators 20,299  linked 20,299  unmatched 0  -> cohort_2022.parquet 114 MB


2023: births 3,605,081  deaths in numerators 19,772  linked 19,772  unmatched 0  -> cohort_2023.parquet 112 MB


In [8]:
import duckdb
duckdb.sql(f"""SELECT cohort, COUNT(*) AS rows, SUM(death::INT)::BIGINT AS deaths,
                     SUM((CAST(RESTATUS AS INT) < 4)::INT)::BIGINT AS resident_births
              FROM read_parquet('{OUT_DIR}/cohort_*.parquet', union_by_name = true) GROUP BY cohort ORDER BY cohort""").pl()

cohort,rows,deaths,resident_births
i16,i64,i64,i64
2016,3956112,22927,3945875
2017,3864754,22197,3855500
2018,3801534,21379,3791712
2019,3757582,20681,3747540
2020,3619826,19376,3613647
2021,3669928,19982,3664292
2022,3676029,20299,3667758
2023,3605081,19772,3596017


**Result.** The table shows the rows (all births, including foreign residents), deaths and resident births in each Parquet file. `01_benchmark` compares these numbers with CDC's own tables.

## Summary

- **Main finding.** All 8 cohort ZIPs are complete and correct: every death joins to a birth, and the record layouts of all years are parsed and checked.
- **Outputs.** One Parquet file per cohort with all public fields under 2023 names, `reference/columns.csv` and `reference/layout_differences.md`.
- **Open issues.** None for the data build.
- **Next notebook.** `01_benchmark.ipynb`.